In [ ]:
import numpy as np 

In [14]:
def fit_stokes_from_window(theta_rad, I_theta):
    """
    Reconstruct the Stokes vector from one angular/temporal window.

    The measured curve is fitted to the harmonic model:

        I(theta) = a0 + a2*sin(2theta) + a4c*cos(4theta) + a4s*sin(4theta)

    and the Stokes parameters are recovered through:

        S1 = 4 a4c
        S2 = 4 a4s
        S3 = -2 a2
        S0 = 2 a0 - 2 a4c

    Parameters
    ----------
    theta_rad : array
        QWP angles inside the selected window.
    I_theta : array
        Intensity values measured in the same window.

    Returns
    -------
    S_fit : ndarray
        Reconstructed Stokes vector [S0, S1, S2, S3].
    coeffs : ndarray
        Fitted harmonic coefficients [a0, a2, a4c, a4s].
    """
    # Build the linear design matrix for least-squares fitting
    A = np.column_stack([
        np.ones_like(theta_rad),
        np.sin(2.0 * theta_rad),
        np.cos(4.0 * theta_rad),
        np.sin(4.0 * theta_rad),
    ])


    # Solve the linear least-squares problem
    coeffs, *_ = np.linalg.lstsq(A, I_theta, rcond=None)
    a0, a2, a4c, a4s = coeffs

    # Convert harmonic coefficients into the Stokes parameters
    S1 = 4.0 * a4c
    S2 = 4.0 * a4s
    S3 = -2.0 * a2
    S0 = 2.0 * a0 - 2.0 * a4c

    print('Coefficients S0, S1, S2, S3: \n', S0, S1, S2, S3)

    return np.array([S0, S1, S2, S3]), coeffs

In [19]:
def normalize_stokes(S):
    """
    Normalize the Stokes vector and compute the degree of polarization.

    Parameters
    ----------
    S : array-like
        Stokes vector [S0, S1, S2, S3].

    Returns
    -------
    s : ndarray
        Normalized Stokes vector [s1, s2, s3].
    dop : float
        Degree of polarization.
    """
    S0, S1, S2, S3 = S
    s = np.array([S1 / S0, S2 / S0, S3 / S0])
    dop = np.sqrt(S1**2 + S2**2 + S3**2) / S0
    return s, dop